# 09 — Level-2 support sensitivity (final)

Final source-specific **flat Level-2** experiment for Papers with Code and bio.tools.

Scientific protocol:
- support eligibility is derived from **training only**;
- thresholds: 20, 10, 5 (execution order chosen to reuse completed results early);
- the eligible vocabulary is frozen and applied unchanged to train/validation/test;
- records with zero eligible labels are removed only **after** filtering;
- primary model: TF-IDF + one-vs-rest LinearSVC;
- Macro-F1 is computed over the complete training-eligible vocabulary with `zero_division=0`.

Engineering fix in this revision: LinearSVC classifiers are checkpointed **one label at a time**. A Colab interruption therefore loses at most one classifier instead of the old 100-classifier block. Training is intentionally memory-safe and does not spawn process workers that duplicate the large sparse TF-IDF matrix.


In [ ]:
from pathlib import Path
import csv, hashlib, json, os, platform, subprocess, sys, time
import numpy as np
import scipy
import sklearn
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score
import joblib

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

SEED = 42
np.random.seed(SEED)

PROJECT_ROOT = Path('/content/drive/MyDrive/phd_research_software')
DATA_ROOT = PROJECT_ROOT / 'data/final/paper_v1'
RESULTS_ROOT = PROJECT_ROOT / 'results'
LEVEL2_FILE = DATA_ROOT / 'level2_fine_grained.jsonl'
MANIFEST_FILE = DATA_ROOT / 'MANIFEST.json'

REPO = Path('/content/research_software_classification_attributes')
if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', 'data-finalization'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', 'data-finalization'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'data-finalization'], check=True)
else:
    subprocess.run([
        'git', 'clone', '-b', 'data-finalization',
        'https://github.com/kuefmz/research_software_classification_attributes.git',
        str(REPO)
    ], check=True)
sys.path.insert(0, str(REPO))

from src.experiments.core import assign_components, audit_split, split_hash
from src.experiments.io import iter_jsonl
from src.experiments.level2 import resumable_ovr_linearsvc_scores

print('python:', sys.version.split()[0])
print('numpy:', np.__version__)
print('scipy:', scipy.__version__)
print('scikit-learn:', sklearn.__version__)
print('repo commit:', subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'], text=True).strip())
assert LEVEL2_FILE.exists(), LEVEL2_FILE
assert MANIFEST_FILE.exists(), MANIFEST_FILE


In [ ]:
manifest = json.loads(MANIFEST_FILE.read_text(encoding='utf-8'))

def manifest_file_info(manifest, filename):
    # Accept the manifest shapes used by the project's dataset-finalization notebooks.
    candidates = []
    if isinstance(manifest, dict):
        for key in ('files', 'artifacts', 'datasets'):
            value = manifest.get(key)
            if isinstance(value, dict) and filename in value:
                candidates.append(value[filename])
            elif isinstance(value, list):
                candidates.extend(x for x in value if isinstance(x, dict) and x.get('name') == filename)
        if filename in manifest and isinstance(manifest[filename], dict):
            candidates.append(manifest[filename])
    return candidates[0] if candidates else {}

level2_manifest = manifest_file_info(manifest, LEVEL2_FILE.name)
DATASET_SHA = (
    level2_manifest.get('sha256')
    or level2_manifest.get('sha_256')
    or level2_manifest.get('hash')
    or manifest.get('level2_sha256')
)
DATASET_ROWS = (
    level2_manifest.get('rows')
    or level2_manifest.get('row_count')
    or manifest.get('level2_count')
)

if not DATASET_SHA:
    print('Manifest does not expose the Level-2 SHA in a recognized field; computing it once.')
    h = hashlib.sha256()
    with LEVEL2_FILE.open('rb') as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b''):
            h.update(chunk)
    DATASET_SHA = h.hexdigest()

print('Level-2 dataset:', LEVEL2_FILE)
print('dataset SHA-256:', DATASET_SHA)
print('manifest row count:', DATASET_ROWS)


## Dependency-aware Level-2 split

The notebook first reuses an existing Level-2 group-aware split if present. If no split artifact exists, it deterministically creates the dependency-component split once using seed 42 and persists it to Drive. The same split is then reused for every source and support threshold.


In [ ]:
SPLIT_DIR = PROJECT_ROOT / 'splits'
SPLIT_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_FILE = SPLIT_DIR / 'level2_fine_grained_group_aware_seed42.csv'
SPLIT_META = SPLIT_DIR / 'level2_fine_grained_group_aware_seed42.json'

# We keep only the fields needed to establish/reuse the split in memory here.
split_records = []
for r in iter_jsonl(LEVEL2_FILE):
    split_records.append({
        'canonical_record_id': str(r['canonical_record_id']),
        'software_group_id': r.get('software_group_id'),
        'repository_group_id': r.get('repository_group_id'),
        'publication_group_id': r.get('publication_group_id'),
    })

if DATASET_ROWS is not None:
    assert len(split_records) == int(DATASET_ROWS), (len(split_records), DATASET_ROWS)

if SPLIT_FILE.exists():
    with SPLIT_FILE.open(newline='', encoding='utf-8') as f:
        split_rows = list(csv.DictReader(f))
    by_id = {r['canonical_record_id']: r for r in split_rows}
    if set(by_id) != {r['canonical_record_id'] for r in split_records}:
        raise ValueError('Existing Level-2 split IDs do not match the final Level-2 dataset')
    # Reconstruct the standard SplitAssignment objects for audit/hash.
    from src.experiments.core import SplitAssignment
    assignments = [
        SplitAssignment(r['canonical_record_id'], r['component_id'], r['partition'])
        for r in split_rows
    ]
    print('Reusing existing Level-2 split:', SPLIT_FILE)
else:
    assignments = assign_components(split_records, seed=SEED)
    with SPLIT_FILE.open('w', newline='', encoding='utf-8') as f:
        w = csv.writer(f)
        w.writerow(['canonical_record_id', 'component_id', 'partition'])
        for a in assignments:
            w.writerow([a.canonical_record_id, a.component_id, a.partition])
    print('Created and persisted Level-2 split:', SPLIT_FILE)

split_audit = audit_split(split_records, assignments)
if not split_audit['passed']:
    raise RuntimeError(f'Level-2 split leakage audit failed: {split_audit}')
SPLIT_SHA = split_hash(assignments)
partition_by_id = {a.canonical_record_id: a.partition for a in assignments}
partition_counts = split_audit['partition_counts']
SPLIT_META.write_text(json.dumps({
    'dataset_sha': DATASET_SHA,
    'dataset_rows': len(split_records),
    'split_sha': SPLIT_SHA,
    'seed': SEED,
    'audit': split_audit,
}, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print('split SHA:', SPLIT_SHA)
print('partition counts:', partition_counts)
print('leakage violations:', split_audit['leakage_violation_count'])
del split_records


## Compact source loading and safe text construction

Target eligibility and the retained record set are computed before fitting TF-IDF. `repository_keywords` / GitHub Topics and all target-derived fields are excluded from the predictor text. Native `repository_description` remains part of safe descriptive repository metadata.


In [ ]:
SAFE_TEXT_FIELDS = (
    'paper_title',
    'paper_abstract',
    'software_name',
    'software_description',
    'repository_title',
    'repository_description',
    'readme_content',
    'somef_description',
)

def clean_text(value):
    if value is None:
        return ''
    if isinstance(value, str):
        return ' '.join(value.split())
    if isinstance(value, (list, tuple, set)):
        return ' '.join(filter(None, (clean_text(v) for v in value)))
    return ' '.join(str(value).split())

def combined_safe_text(record):
    chunks = []
    for field in SAFE_TEXT_FIELDS:
        value = clean_text(record.get(field))
        if value:
            chunks.append(f'{field}: {value}')
    return '\n'.join(chunks)

def load_source_records(source):
    # Deliberately compact: do not retain README/abstract text for all ~140k PwC
    # rows in Python objects. Text is streamed from JSONL only if a TF-IDF cache
    # must actually be built.
    parts = {'train': [], 'validation': [], 'test': []}
    for r in iter_jsonl(LEVEL2_FILE):
        if r.get('source') != source:
            continue
        rid = str(r['canonical_record_id'])
        part = partition_by_id.get(rid)
        if part not in parts:
            raise ValueError(f'Missing Level-2 split partition for {rid}')
        labels = tuple(str(x) for x in (r.get('target_labels') or []))
        if labels:
            parts[part].append({
                'canonical_record_id': rid,
                'labels': labels,
            })
    return parts

def filter_for_threshold(parts, threshold):
    from collections import Counter
    support = Counter(label for r in parts['train'] for label in r['labels'])
    eligible = sorted(label for label, n in support.items() if n >= threshold)
    eligible_set = set(eligible)
    out = {}
    for part, rows in parts.items():
        kept = []
        for r in rows:
            labels = tuple(x for x in r['labels'] if x in eligible_set)
            if labels:
                kept.append({
                    'canonical_record_id': r['canonical_record_id'],
                    'labels': labels,
                })
        out[part] = kept
    return out, eligible, {label: int(support[label]) for label in eligible}

def partition_summary(parts):
    return {
        part: {
            'records': len(rows),
            'assignments': int(sum(len(r['labels']) for r in rows)),
        }
        for part, rows in parts.items()
    }

def stream_texts_for_ids(source, ordered_rows):
    # load_source_records appended each partition in source-file order, so
    # a second JSONL stream yields the retained IDs in the same order while
    # keeping only one record's large text fields resident at a time.
    wanted = [r['canonical_record_id'] for r in ordered_rows]
    wanted_set = set(wanted)
    seen = 0
    expected_pos = 0
    for record in iter_jsonl(LEVEL2_FILE):
        rid = str(record['canonical_record_id'])
        if record.get('source') != source or rid not in wanted_set:
            continue
        if rid != wanted[expected_pos]:
            raise ValueError(
                f'Predictor text order drift: expected {wanted[expected_pos]}, got {rid}'
            )
        expected_pos += 1
        seen += 1
        yield combined_safe_text(record)
    if seen != len(wanted):
        raise ValueError(f'Missing predictor text records: expected {len(wanted)}, found {seen}')


## TF-IDF cache

Each source×support-threshold view has a Drive-backed TF-IDF cache. The cache is reused only when dataset SHA, split SHA, source, threshold, eligible-label order, and retained record IDs match.


In [ ]:
def ids_hash(rows):
    h = hashlib.sha256()
    for r in rows:
        h.update(r['canonical_record_id'].encode('utf-8'))
        h.update(b'\n')
    return h.hexdigest()

def atomic_json(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(payload, indent=2, sort_keys=True) + '\n', encoding='utf-8')
    os.replace(tmp, path)

def load_or_build_tfidf(run_dir, source, threshold, filtered, eligible):
    cache_dir = run_dir / 'checkpoints' / 'tfidf'
    cache_dir.mkdir(parents=True, exist_ok=True)
    meta_file = cache_dir / 'meta.json'
    matrix_files = {p: cache_dir / f'{p}.npz' for p in ('train','validation','test')}
    vectorizer_file = cache_dir / 'vectorizer.joblib'

    expected = {
        'dataset_sha': DATASET_SHA,
        'split_sha': SPLIT_SHA,
        'source': source,
        'threshold': int(threshold),
        'eligible_labels': eligible,
        'record_id_hashes': {p: ids_hash(filtered[p]) for p in filtered},
        'record_counts': {p: len(filtered[p]) for p in filtered},
        'tfidf': {
            'ngram_range': [1, 2],
            'sublinear_tf': True,
            'min_df': 2,
            'max_features': 50000,
            'vocabulary_fit': 'training_only',
        },
    }

    reusable = False
    if meta_file.exists() and vectorizer_file.exists() and all(p.exists() and p.stat().st_size for p in matrix_files.values()):
        try:
            saved = json.loads(meta_file.read_text(encoding='utf-8'))
            reusable = saved == expected
        except Exception:
            reusable = False

    if reusable:
        print('TF-IDF CACHE: validated cache found; loading sparse matrices from Drive')
        x = {p: sparse.load_npz(matrix_files[p]).tocsr() for p in matrix_files}
        vectorizer = joblib.load(vectorizer_file)
    else:
        print('TF-IDF CACHE: missing/stale; fitting vocabulary on TRAIN only')
        vectorizer = TfidfVectorizer(
            ngram_range=(1,2),
            sublinear_tf=True,
            min_df=2,
            max_features=50000,
            dtype=np.float32,
        )
        x = {}
        x['train'] = vectorizer.fit_transform(stream_texts_for_ids(source, filtered['train'])).tocsr()
        x['validation'] = vectorizer.transform(stream_texts_for_ids(source, filtered['validation'])).tocsr()
        x['test'] = vectorizer.transform(stream_texts_for_ids(source, filtered['test'])).tocsr()
        for p in x:
            sparse.save_npz(matrix_files[p], x[p], compressed=True)
        joblib.dump(vectorizer, vectorizer_file)
        atomic_json(meta_file, expected)
        # Read-back gate
        assert json.loads(meta_file.read_text(encoding='utf-8')) == expected

    for p in x:
        if x[p].shape[0] != len(filtered[p]):
            raise ValueError(f'TF-IDF row mismatch for {p}: {x[p].shape[0]} vs {len(filtered[p])}')
    print('TF-IDF MATRIX SHAPES:', {p: x[p].shape for p in x})
    print('TF-IDF VOCABULARY SIZE:', len(vectorizer.vocabulary_))
    return x, vectorizer


## Resumable flat Level-2 experiment

The key fix is here: `resumable_ovr_linearsvc_scores` saves validation/test decision scores for **every completed label** under `checkpoints/label_scores/`. It validates those files before reuse and prints progress after every newly completed classifier.

`dual="auto"` is used so LinearSVC can choose the appropriate primal/dual optimization for the matrix dimensions; the SVM objective/model family is unchanged. `max_iter=5000` makes the convergence cap explicit and convergence status is recorded per label.


In [ ]:
def result_is_valid(metrics_file, *, source, threshold):
    if not metrics_file.exists() or metrics_file.stat().st_size == 0:
        return False
    try:
        result = json.loads(metrics_file.read_text(encoding='utf-8'))
    except Exception:
        return False
    return (
        result.get('status') == 'COMPLETED_VALIDATED'
        and result.get('dataset_sha') == DATASET_SHA
        and result.get('split_sha') == SPLIT_SHA
        and result.get('source') == source
        and int(result.get('support_threshold', -1)) == int(threshold)
        and result.get('model') == 'TF-IDF + One-vs-Rest LinearSVC'
    )

def run_flat(source, threshold, source_parts):
    source_dir = 'bio_tools' if source == 'bio.tools' else 'papers_with_code'
    run_dir = RESULTS_ROOT / 'level2' / 'flat' / source_dir / f'support_{threshold}'
    run_dir.mkdir(parents=True, exist_ok=True)
    metrics_file = run_dir / 'metrics.json'

    print('\n' + '='*88)
    print('SOURCE:', source, 'SUPPORT THRESHOLD:', threshold)

    if result_is_valid(metrics_file, source=source, threshold=threshold):
        print('VALIDATED RESULT REUSED:', metrics_file)
        return json.loads(metrics_file.read_text(encoding='utf-8'))

    filtered, eligible, train_support = filter_for_threshold(source_parts, threshold)
    summary = partition_summary(filtered)
    print('ELIGIBLE LABELS:', len(eligible))
    print('TRAIN RECORDS:', summary['train']['records'])
    print('VALIDATION RECORDS:', summary['validation']['records'])
    print('TEST RECORDS:', summary['test']['records'])
    print('TRAIN ASSIGNMENTS:', summary['train']['assignments'])
    print('VALIDATION ASSIGNMENTS:', summary['validation']['assignments'])
    print('TEST ASSIGNMENTS:', summary['test']['assignments'])

    if not eligible:
        raise RuntimeError(f'No eligible labels for {source} support >= {threshold}')

    mlb = MultiLabelBinarizer(classes=eligible, sparse_output=True)
    mlb.fit([eligible])
    y = {p: mlb.transform([r['labels'] for r in filtered[p]]).tocsr() for p in filtered}

    x, vectorizer = load_or_build_tfidf(run_dir, source, threshold, filtered, eligible)

    classifier_checkpoint_dir = run_dir / 'checkpoints'
    val_scores, test_scores, classifier_meta = resumable_ovr_linearsvc_scores(
        x_train=x['train'],
        y_train=y['train'],
        x_validation=x['validation'],
        x_test=x['test'],
        labels=eligible,
        checkpoint_dir=classifier_checkpoint_dir,
        random_state=SEED,
        class_weight='balanced',
        max_iter=5000,
        tol=1e-4,
        progress_every=1,
    )

    # Flat LinearSVC decision rule. No test information is used for model/label selection.
    decision_threshold = 0.0
    y_pred_test = (test_scores >= decision_threshold).astype(np.int8)
    y_true_test = y['test'].toarray().astype(np.int8, copy=False)

    metrics = {
        'macro_f1': float(f1_score(y_true_test, y_pred_test, average='macro', zero_division=0)),
        'micro_f1': float(f1_score(y_true_test, y_pred_test, average='micro', zero_division=0)),
        'weighted_f1': float(f1_score(y_true_test, y_pred_test, average='weighted', zero_division=0)),
    }

    # Save prediction matrices for downstream error analysis/bootstrap without retraining.
    pred_dir = run_dir / 'predictions'
    pred_dir.mkdir(parents=True, exist_ok=True)
    sparse.save_npz(pred_dir / 'test_true.npz', sparse.csr_matrix(y_true_test), compressed=True)
    sparse.save_npz(pred_dir / 'test_pred.npz', sparse.csr_matrix(y_pred_test), compressed=True)
    (pred_dir / 'test_ids.json').write_text(
        json.dumps([r['canonical_record_id'] for r in filtered['test']]) + '\n',
        encoding='utf-8'
    )

    per_label = []
    for j, label in enumerate(eligible):
        yt = y_true_test[:, j]
        yp = y_pred_test[:, j]
        per_label.append({
            'label': label,
            'train_support': int(train_support[label]),
            'validation_support': int(y['validation'][:, j].sum()),
            'test_support': int(yt.sum()),
            'f1': float(f1_score(yt, yp, zero_division=0)),
            'classifier_n_iter': int(classifier_meta[j]['n_iter']),
            'classifier_converged': bool(classifier_meta[j]['converged']),
        })
    with (run_dir / 'per_label.csv').open('w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=list(per_label[0]))
        w.writeheader()
        w.writerows(per_label)

    result = {
        'status': 'COMPLETED_VALIDATED',
        'dataset_sha': DATASET_SHA,
        'dataset_row_count': int(DATASET_ROWS) if DATASET_ROWS is not None else None,
        'source': source,
        'task': 'level2_flat_multilabel_support_sensitivity',
        'split_sha': SPLIT_SHA,
        'seed': SEED,
        'support_threshold': int(threshold),
        'representation': 'COMBINED_SAFE',
        'safe_text_fields': list(SAFE_TEXT_FIELDS),
        'model': 'TF-IDF + One-vs-Rest LinearSVC',
        'model_configuration': {
            'class_weight': 'balanced',
            'random_state': SEED,
            'max_iter': 5000,
            'tol': 1e-4,
            'dual': 'auto',
        },
        'tfidf_configuration': {
            'ngram_range': [1,2],
            'sublinear_tf': True,
            'min_df': 2,
            'max_features': 50000,
            'training_only_vocabulary': True,
        },
        'decision_threshold': decision_threshold,
        'eligible_label_count': len(eligible),
        'eligible_labels': eligible,
        'train_support': train_support,
        'train_records': summary['train']['records'],
        'validation_records': summary['validation']['records'],
        'test_records': summary['test']['records'],
        'train_assignments': summary['train']['assignments'],
        'validation_assignments': summary['validation']['assignments'],
        'test_assignments': summary['test']['assignments'],
        'metrics': metrics,
        'macro_f1_universe': 'full training-eligible vocabulary',
        'zero_division': 0,
        'non_converged_classifier_count': int(sum(not m['converged'] for m in classifier_meta)),
        'code_commit': subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'], text=True).strip(),
        'timestamp_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    }
    atomic_json(metrics_file, result)

    # Read-back validation: a partially-written/corrupt result never counts as complete.
    reread = json.loads(metrics_file.read_text(encoding='utf-8'))
    assert reread['status'] == 'COMPLETED_VALIDATED'
    assert reread['dataset_sha'] == DATASET_SHA
    assert reread['split_sha'] == SPLIT_SHA
    assert reread['eligible_label_count'] == len(eligible)
    assert set(reread['metrics']) == {'macro_f1','micro_f1','weighted_f1'}
    print('COMPLETED_VALIDATED:', metrics_file)
    print('METRICS:', metrics)
    return reread


In [ ]:
RUN_ORDER = [
    ('bio.tools', 20),
    ('bio.tools', 10),
    ('bio.tools', 5),
    ('papers_with_code', 20),
    ('papers_with_code', 10),
    ('papers_with_code', 5),
]

source_cache = {}
results = []
for source, threshold in RUN_ORDER:
    if source not in source_cache:
        print('\nLOADING SOURCE PARTITIONS ONCE FOR:', source)
        source_cache[source] = load_source_records(source)
        print('SOURCE PARTITION COUNTS:', {p: len(v) for p, v in source_cache[source].items()})
    results.append(run_flat(source, threshold, source_cache[source]))

summary_file = RESULTS_ROOT / 'level2' / 'flat' / 'level2_support_sensitivity_summary.json'
atomic_json(summary_file, {
    'status': 'COMPLETED_VALIDATED',
    'dataset_sha': DATASET_SHA,
    'split_sha': SPLIT_SHA,
    'results': results,
})
print('\nNOTEBOOK STATUS: COMPLETED_VALIDATED')
print('Summary:', summary_file)


## Optional hierarchy-aware sensitivity

The hierarchy-aware experiment remains separate from the flat primary analysis. It should be executed only after the flat results above are complete and only where the source-specific parent relationships can be applied without inventing a taxonomy. No hierarchy-aware result is required for completion of the flat Level-2 primary experiment.
